<a href="https://colab.research.google.com/github/ITSamurai7/projeto-machine-learning-ideb/blob/feat%2Fbaseline/etapas_1_e_3_baseline_organizado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Configuração Inicial: Preparando a Base do IDEB (INEP)
**Contexto:** Antes de iniciarmos nossa Análise Exploratória focada nos fatores de vulnerabilidade e evasão escolar, precisamos carregar os dados e criar nossa variável principal de estudo: o `alerta_risco`. Essa classificação nos ajudará a mapear os municípios que apresentam falhas críticas no ensino de Matemática.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from google.colab import drive

# Montando o Drive e lendo os dados
drive.mount('/content/drive')
CAMINHO = '/content/drive/MyDrive/machine_learning/ideb_ensino_medio.csv'
df = pd.read_csv(CAMINHO, encoding='utf-8-sig')

# Engenharia de Atributos Auxiliares (Região)
uf_para_regiao = {
    'AC': 'Norte', 'AL': 'Nordeste', 'AP': 'Norte', 'AM': 'Norte', 'BA': 'Nordeste',
    'CE': 'Nordeste', 'DF': 'Centro-Oeste', 'ES': 'Sudeste', 'GO': 'Centro-Oeste',
    'MA': 'Nordeste', 'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste', 'MG': 'Sudeste',
    'PA': 'Norte', 'PB': 'Nordeste', 'PR': 'Sul', 'PE': 'Nordeste', 'PI': 'Nordeste',
    'RJ': 'Sudeste', 'RN': 'Nordeste', 'RS': 'Sul', 'RO': 'Norte', 'RR': 'Norte',
    'SC': 'Sul', 'SP': 'Sudeste', 'SE': 'Nordeste', 'TO': 'Norte'
}
df['regiao'] = df['sigla_uf'].map(uf_para_regiao)

# Regra de Negócio: Zona de Alto Risco em Matemática (< 250 pts OU Rendimento < 0.75)
df['alerta_risco'] = np.where(
    (df['nota_saeb_matematica'] < 250) | (df['indicador_rendimento'] < 0.75),
    'Alto Risco', 'Estável / Regular'
)

print(f"Base carregada! Total de registros: {df.shape[0]}")

Mounted at /content/drive
Base carregada! Total de registros: 27769


# Etapa 1: O Raio-X da Matemática no Ensino Médio
**Objetivo:** O ritual de reconhecimento da nossa variável principal. Vamos extrair as estatísticas descritivas rigorosas (usando o desvio padrão amostral correto) e visualizar a distribuição da proficiência, identificando onde está a linha crítica que antecede o abandono escolar.

In [ ]:
def gerar_resumo_estatistico(serie, nome_var):
    s = serie.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    dp_amostral = s.std(ddof=1) # Usando ddof=1 (Pandas default) conforme Lab 2
    media = s.mean()

    return pd.DataFrame({
        'Variável': [nome_var],
        'N': [len(s)],
        'Média': [round(media, 2)],
        'Mediana (Q2)': [round(s.median(), 2)],
        'Q1 (25%)': [round(q1, 2)],
        'Q3 (75%)': [round(q3, 2)],
        'IQR': [round(q3 - q1, 2)],
        'Desvio Padrão': [round(dp_amostral, 2)],
        'CV (%)': [round((dp_amostral / media) * 100, 2)]
    })

display(gerar_resumo_estatistico(df['nota_saeb_matematica'], 'Nota SAEB Matemática'))

fig1 = px.histogram(
    df.dropna(subset=['nota_saeb_matematica']), x='nota_saeb_matematica', nbins=40,
    title='<b>Distribuição da Proficiência em Matemática (Ensino Médio)</b>',
    labels={'nota_saeb_matematica': 'Nota SAEB Matemática (0-500)', 'count': 'Qtd. de Municípios'},
    template='plotly_white', color_discrete_sequence=['#1f77b4']
)
fig1.add_vline(x=250, line_dash="dash", line_color="red", annotation_text="Corte Crítico (<250)")
fig1.show()

,Variável,N,Média,Mediana (Q2),Q1 (25%),Q3 (75%),IQR,Desvio Padrão,CV (%)
0,Nota SAEB Matemática,23921,268.78,268.56,255.42,281.39,25.97,19.95,7.42


**Análise da Distribuição e Estatística Descritiva da Nota de Matemática:**  
A distribuição da proficiência em matemática apresenta uma média de **268,78 pontos** e uma mediana de **268,56 pontos**. A proximidade entre centro e mediana indica uma simetria geral, mas com dispersão relevante: o desvio padrão amostral ($ddof=1$) é de **19,95 pontos** (Coeficiente de Variação de **7,42%**).
O achado mais crítico reside no primeiro quartil ($Q1 = 255,42$). Isso demonstra que **mais de 25% dos municípios registraram nota muito próxima ou inferior ao limite crítico de 250 pontos** ("Abaixo do Básico"). A linha tracejada vermelha expõe visualmente a grande massa de alunos na faixa de vulnerabilidade que antecede a evasão escolar por frustração pedagógica.

# Etapa 2: Dispersão Regional e a Busca por Outliers
**Objetivo:** Médias agregadas costumam mentir. Vamos quebrar os dados por região geográfica e aplicar a regra das Cercas de Tukey (1,5x IQR) para identificar os "pontos fora da curva" reais — municípios de sucesso atípico ou fracasso severo.

In [ ]:
fig2 = px.box(
    df.dropna(subset=['nota_saeb_matematica']), x='regiao', y='nota_saeb_matematica',
    color='regiao', points='outliers',
    title='<b>Dispersão e Outliers da Nota de Matemática por Região</b>',
    labels={'regiao': 'Região do Brasil', 'nota_saeb_matematica': 'Nota Matemática'},
    template='plotly_white'
)
fig2.add_hline(y=250, line_dash="dot", line_color="red", annotation_text="Vulnerabilidade (250 pts)")
fig2.update_layout(showlegend=False)
fig2.show()

**Análise de Dispersão Regional e Identificação de Outliers pela Regra de Tukey:**  
A análise por boxplot explicita a severa disparidade do sistema educacional brasileiro. A Região Sul lidera com mediana de **280,00 pontos**, enquanto a Região Norte apresenta a mediana mais baixa (**252,04 pontos**), com a base da sua caixa ($Q1$) posicionada diretamente na zona de risco de evasão.
Aplicando a Regra de Tukey ($Q1 - 1,5 \times IQR$ e $Q3 + 1,5 \times IQR$), o algoritmo identificou pontos soltos (outliers). Temos **42 outliers inferiores** (municípios em colapso educacional) e **216 outliers superiores** (sobretudo no Nordeste e Sudeste, representando ilhas de excelência atípicas). Preservaremos esses dados, pois representam fenômenos reais e não erros de digitação.

# Etapa 3: Matriz de Correlação e Falsas Premissas
**Objetivo:** O que realmente impacta a nota? Vamos calcular o coeficiente linear de Pearson ($r$) para desvendar se taxa de aprovação formal e o tamanho da população caminham junto com o aprendizado ou se contam histórias separadas.

In [ ]:
colunas_num = ['nota_saeb_matematica', 'nota_saeb_lingua_portuguesa', 'indicador_rendimento', 'taxa_aprovacao', 'ideb', 'populacao']
matriz_corr = df[colunas_num].corr(method='pearson').round(2)

fig3 = px.imshow(
    matriz_corr, text_auto=True, aspect="auto", zmin=-1, zmax=1,
    color_continuous_scale='RdBu_r',
    title='<b>Matriz de Correlação Linear de Pearson</b>',
    labels=dict(x="Variáveis", y="Variáveis", color="r de Pearson")
)
fig3.show()

**Análise da Correlação Linear de Pearson ($r$):**  
A matriz revela uma correlação positiva quase perfeita entre Matemática e Língua Portuguesa ($r = 0,90$), mostrando que a proficiência em linguagem e lógica evoluem juntas.
O grande alerta analítico recai sobre a correlação muito fraca entre a Nota de Matemática e a **Taxa de Aprovação ($r = 0,21$)** ou **Indicador de Rendimento ($r = 0,22$)**. Isso comprova matematicamente que aprovar o aluno não significa que ele aprendeu. Além disso, a `populacao` apresenta $r = 0,00$ com o desempenho, indicando que municípios pequenos ou grandes possuem desafios de evasão semelhantes que não dependem do seu tamanho linearmente.

# Etapa 4: O Paradoxo do Rendimento (A Aprovação Mascarada)
**Objetivo:** O gráfico que corrobora a matriz de correlação. Vamos cruzar as notas com o indicador de fluxo para flagrar os municípios que "aprovam no papel, mas não ensinam na prática" (Alto Risco).

In [ ]:
fig4 = px.scatter(
    df.dropna(subset=['nota_saeb_matematica', 'indicador_rendimento']),
    x='indicador_rendimento', y='nota_saeb_matematica',
    color='alerta_risco', opacity=0.6,
    color_discrete_map={'Alto Risco': '#d62728', 'Estável / Regular': '#2ca02c'},
    hover_data=['municipio', 'sigla_uf', 'taxa_aprovacao'],
    title='<b>Rendimento Escolar vs. Proficiência em Matemática (Zonas Mascaradas)</b>',
    labels={'indicador_rendimento': 'Indicador de Rendimento (Fluxo)'},
    template='plotly_white'
)
fig4.add_hline(y=250, line_dash="dash", line_color="black")
fig4.add_vline(x=0.85, line_dash="dash", line_color="black")
fig4.show()

**Diagnóstico de Vulnerabilidade e Aprovação Mascarada (Dispersão):**  
O quadrante inferior direito (Rendimento superior a $0,85$ e nota de matemática $< 250$, em vermelho) expõe a **Aprovação Mascarada**. Nesses municípios, o fluxo escolar caminha artificialmente bem, o que cria uma falsa sensação de segurança contra a evasão, enquanto o aprendizado real beira o analfabetismo funcional em números. Essa falha oculta justifica a necessidade do nosso projeto focar no cruzamento de dados: prever o abandono escolar requer muito mais do que olhar apenas se os alunos estão sendo "aprovados" administrativamente.

# Etapa 5: O Mapa da Vulnerabilidade Escolar (Efeito de Composição)
**Objetivo:** Agrupar e quantificar. Qual a real proporção da crise e das zonas de alto risco quando isolamos geograficamente o problema?

In [ ]:
df_risco_regiao = (pd.crosstab(df['regiao'], df['alerta_risco'], normalize='index') * 100).reset_index()

fig5 = px.bar(
    df_risco_regiao, x='regiao', y='Alto Risco', text_auto='.1f',
    title='<b>Porcentagem de Municípios em Situação de Alto Risco por Região (%)</b>',
    labels={'regiao': 'Região', 'Alto Risco': '% Municípios em Alto Risco'},
    color='Alto Risco', color_continuous_scale='Reds', template='plotly_white'
)
fig5.update_layout(coloraxis_showscale=False)
fig5.show()

**Análise do Percentual de Alto Risco por Região (Efeito de Composição):**  
A agregação do risco revela o efeito de composição territorial. A Região Norte desponta com o percentual mais grave (**42,6% de seus municípios em Alto Risco de retenção/evasão severa**), seguida do Nordeste (**34,2%**). Em forte discrepância, a Região Sudeste registra apenas **7,1%**.
Para a próxima etapa de pré-processamento, essa disparidade indica que não podemos treinar o modelo globalmente de forma cega; o algoritmo supervisionado de `Machine Learning` precisará receber a coluna "Região" codificada como variável preditora, para não subestimar o risco de evasão estrutural que afeta o Norte e Nordeste.

---
# Etapa 6: Protocolo Experimental e Modelo de Referência (Baseline)

**Objetivo:** Definir *como* os modelos serão avaliados (protocolo experimental) e executar um modelo simples de referência (baseline), com a métrica escolhida em função do **custo do erro** no domínio.

**Roteiro desta etapa**

| Tópico | O que será feito |
|---|---|
| 6.1 Diagnóstico da base | Conferir colunas, ausentes, anos e repetição de municípios |
| 6.2 Preparação da base | Filtrar registros com alvo incompleto e codificar o alvo em 0/1 |
| 6.3 Variáveis e cenários | Definir quais colunas entram no modelo (cenários A e B) |
| 6.4 Divisão treino/teste | Separar os dados por município, de forma estratificada |
| 6.5 Métrica | Escolher a métrica pelo custo do erro |
| 6.6 Baseline e validação cruzada | Treinar o modelo de referência e compará-lo a um classificador trivial |
| 6.7 Avaliação final | Medir o desempenho no teste (uma única vez) |
| 6.8 Conclusões | Resumir resultados e limitações |

### Bibliotecas e parâmetros

In [2]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, fbeta_score,
                             make_scorer, precision_score, recall_score)
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Semente fixa: garante a mesma divisão e os mesmos resultados a cada execução
SEMENTE = 42

## 6.1 Diagnóstico da base

Antes de definir o protocolo, conferimos o que a base tem de relevante para as decisões seguintes:

- **Ausentes:** nota, IDEB e padronizada faltam nas mesmas linhas; rendimento e aprovação faltam em outras.
- **Colunas de valor único:** não trazem informação para o modelo.
- **Repetição de municípios:** o mesmo município aparece em vários anos, o que afeta a divisão treino/teste. Os municípios são contados por `id_municipio`, pois `municipio` repete nomes entre UFs.

In [3]:
print('=' * 60)
print('DIAGNÓSTICO DA BASE')
print('=' * 60)

print('\n1) COLUNAS')
print('   ' + ', '.join(df.columns))

print('\n2) VALORES AUSENTES (apenas colunas com ausentes)')
ausentes = df.isna().sum()
print(ausentes[ausentes > 0].to_string())

print('\n3) DISTRIBUIÇÃO DO ALVO (base original)')
print(df['alerta_risco'].value_counts(normalize=True).round(3).to_string())

print('\n4) REGISTROS POR ANO')
print(df['ano'].value_counts().sort_index().to_string())

print('\n5) VALORES DISTINTOS EM COLUNAS CANDIDATAS A CONSTANTES (1 = sem informação)')
print(df[['rede', 'ensino', 'anos_escolares']].nunique().to_string())

print('\n6) REPETIÇÃO DE MUNICÍPIOS')
print(f'   Municípios distintos (id_municipio): {df["id_municipio"].nunique()}')
print('   Registros por município:')
print(df.groupby('id_municipio').size().describe().round(2).to_string())

print('\n7) AUSENTES NAS COLUNAS DA REGRA DO ALVO')
sem_nota = df['nota_saeb_matematica'].isna()
sem_rend = df['indicador_rendimento'].isna()
print(f'   Nota e rendimento preenchidos : {(~sem_nota & ~sem_rend).sum()}')
print(f'   Só a nota ausente             : {(sem_nota & ~sem_rend).sum()}')
print(f'   Só o rendimento ausente       : {(~sem_nota & sem_rend).sum()}')
print(f'   Nota e rendimento ausentes    : {(sem_nota & sem_rend).sum()}')

DIAGNÓSTICO DA BASE

1) COLUNAS
   ano, sigla_uf, id_municipio, municipio, rede, ensino, anos_escolares, taxa_aprovacao, indicador_rendimento, nota_saeb_matematica, nota_saeb_lingua_portuguesa, nota_saeb_media_padronizada, ideb, projecao, populacao, regiao, alerta_risco

2) VALORES AUSENTES (apenas colunas com ausentes)
taxa_aprovacao                   419
indicador_rendimento             419
nota_saeb_matematica            3848
nota_saeb_lingua_portuguesa     3848
nota_saeb_media_padronizada     3848
ideb                            3848
projecao                       22491
populacao                          1

3) DISTRIBUIÇÃO DO ALVO (base original)
alerta_risco
Estável / Regular    0.798
Alto Risco           0.202

4) REGISTROS POR ANO
ano
2017    5545
2019    5545
2021    5557
2023    5559
2025    5563

5) VALORES DISTINTOS EM COLUNAS CANDIDATAS A CONSTANTES (1 = sem informação)
rede              1
ensino            1
anos_escolares    1

6) REPETIÇÃO DE MUNICÍPIOS
   Municípios dis

## 6.2 Preparação da base e variável-alvo

O alvo `alerta_risco` foi construído na configuração inicial: **Alto Risco** quando a nota de Matemática é inferior a 250 **ou** o indicador de rendimento é inferior a 0,75.

**Problema:** quando a nota de Matemática está ausente, a comparação `NaN < 250` resulta em falso, e o registro pode ser rotulado como "Estável" sem que a regra tenha sido avaliada de fato. Por isso, removemos os registros em que a nota **ou** o rendimento estão vazios, mantendo apenas rótulos confiáveis.

Em seguida, o alvo é codificado em 0/1, com **1 = Alto Risco** (a classe positiva, a que queremos encontrar).

In [4]:
# Mantém apenas registros em que a regra do alvo pôde ser avaliada por completo
df_ok = df.dropna(subset=['nota_saeb_matematica', 'indicador_rendimento']).copy()

# Alvo binário: 1 = Alto Risco (classe positiva), 0 = Estável / Regular
df_ok['y'] = (df_ok['alerta_risco'] == 'Alto Risco').astype(int)

removidos = len(df) - len(df_ok)
print('=' * 50)
print('PREPARAÇÃO DA BASE')
print('=' * 50)
print(f'   Registros na base original   : {len(df)}')
print(f'   Registros removidos          : {removidos} ({removidos / len(df):.1%})')
print(f'   Registros na base de trabalho: {len(df_ok)}')
print(f'   Municípios distintos         : {df_ok["id_municipio"].nunique()}')
print(f'   Proporção de Alto Risco      : {df_ok["y"].mean():.1%}')

PREPARAÇÃO DA BASE
   Registros na base original   : 27769
   Registros removidos          : 3848 (13.9%)
   Registros na base de trabalho: 23921
   Municípios distintos         : 5558
   Proporção de Alto Risco      : 20.9%


## 6.3 Seleção de variáveis e cenários

**Variáveis excluídas das entradas do modelo**

| Colunas | Motivo |
|---|---|
| `nota_saeb_matematica`, `indicador_rendimento` | Definem o alvo (vazamento de dados) |
| `taxa_aprovacao`, `ideb`, `nota_saeb_media_padronizada` | Derivadas das duas colunas acima |
| `projecao` | Meta oficial, quase toda vazia |
| `rede`, `ensino`, `anos_escolares` | Possuem um único valor (não trazem informação) |
| `municipio`, `id_municipio` | Identificadores (o `id_municipio` é usado apenas para agrupar a divisão) |
| `regiao` | Redundante com `sigla_uf`, que traz mais detalhe |

**Cenários avaliados**

| Cenário | Variáveis | Pergunta que responde |
|---|---|---|
| **A (só contexto)** | `ano`, `sigla_uf`, `populacao` | É possível prever o risco apenas pelo contexto do município? |
| **B (+ português)** | as mesmas, mais `nota_saeb_lingua_portuguesa` | Quanto o modelo melhora ao conhecer a nota de Português? |

A nota de Português é fortemente correlacionada com a de Matemática (que define o alvo). Por isso, o cenário B funciona como um **limite superior** de desempenho, e o cenário A é o resultado "honesto".

In [5]:
# Cada cenário: (colunas numéricas, colunas categóricas)
cenarios = {
    'A (só contexto)': (['ano', 'populacao'], ['sigla_uf']),
    'B (+ português)': (['ano', 'populacao', 'nota_saeb_lingua_portuguesa'], ['sigla_uf']),
}

# Todas as colunas candidatas (cenário B), usadas na divisão e na leitura dos dados
features_B = cenarios['B (+ português)'][0] + cenarios['B (+ português)'][1]

for nome, (cols_num, cols_cat) in cenarios.items():
    print(f'Cenário {nome}: {cols_num + cols_cat}')

Cenário A (só contexto): ['ano', 'populacao', 'sigla_uf']
Cenário B (+ português): ['ano', 'populacao', 'nota_saeb_lingua_portuguesa', 'sigla_uf']


## 6.4 Divisão treino/teste

- **80% treino / 20% teste.** O teste fica reservado para uma **única** avaliação final.
- **Agrupada por município:** o mesmo município aparece em até 5 anos e sua população muda pouco, funcionando como "impressão digital". Se o município estivesse nos dois conjuntos, o modelo acertaria por memorização. Com a divisão agrupada, o teste mede o desempenho em **municípios nunca vistos**.
- **Estratificada pelo alvo:** mantém a proporção de Alto Risco (~21%) nos dois conjuntos.
- **Semente fixa** (`SEMENTE = 42`): a mesma divisão é reproduzida a cada execução.

In [6]:
X = df_ok[features_B]
y = df_ok['y']
grupos = df_ok['id_municipio']

# 5 partes estratificadas e agrupadas; a primeira vira o teste (20%)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
train_idx, test_idx = next(sgkf.split(X, y, groups=grupos))

df_train = df_ok.iloc[train_idx]
df_test = df_ok.iloc[test_idx]
y_train, y_test = df_train['y'], df_test['y']

# ---------- verificações ----------
total = len(df_train) + len(df_test)
municipios_comuns = set(df_train['id_municipio']) & set(df_test['id_municipio'])

print('=' * 50)
print('RESUMO DA DIVISÃO TREINO / TESTE (por município)')
print('=' * 50)

print('\n1) TAMANHO DOS CONJUNTOS')
print(f'   Total (base filtrada) : {total}')
print(f'   Treino                : {len(df_train)} linhas ({len(df_train) / total:.1%})')
print(f'   Teste                 : {len(df_test)} linhas ({len(df_test) / total:.1%})')

print('\n2) PROPORÇÃO DE ALTO RISCO (esperado: ~20,9% nos dois)')
print(f'   Base filtrada         : {y.mean():.1%}')
print(f'   Treino                : {y_train.mean():.1%}')
print(f'   Teste                 : {y_test.mean():.1%}')

print('\n3) MUNICÍPIOS')
print(f'   Municípios no treino  : {df_train["id_municipio"].nunique()}')
print(f'   Municípios no teste   : {df_test["id_municipio"].nunique()}')
print(f'   Municípios nos dois   : {len(municipios_comuns)}  (deve ser 0)')

print('\n' + ('OK: nenhum município aparece nos dois conjuntos.'
              if len(municipios_comuns) == 0
              else 'ATENÇÃO: há municípios repetidos entre treino e teste!'))

RESUMO DA DIVISÃO TREINO / TESTE (por município)

1) TAMANHO DOS CONJUNTOS
   Total (base filtrada) : 23921
   Treino                : 19158 linhas (80.1%)
   Teste                 : 4763 linhas (19.9%)

2) PROPORÇÃO DE ALTO RISCO (esperado: ~20,9% nos dois)
   Base filtrada         : 20.9%
   Treino                : 20.9%
   Teste                 : 20.7%

3) MUNICÍPIOS
   Municípios no treino  : 4449
   Municípios no teste   : 1109
   Municípios nos dois   : 0  (deve ser 0)

OK: nenhum município aparece nos dois conjuntos.


## 6.5 Métrica de avaliação e custo do erro

| | Modelo diz **Alto Risco** | Modelo diz **Estável** |
|---|---|---|
| **Realmente Alto Risco** | Acerto (verdadeiro positivo) | **Falso negativo** |
| **Realmente Estável** | **Falso positivo** | Acerto (verdadeiro negativo) |

- **Falso negativo (erro mais grave):** o município está em situação crítica e o modelo não avisa, então o apoio e o acompanhamento não chegam a quem precisa.
- **Falso positivo (erro recuperável):** o modelo aponta um município que está bem, o que gera apenas uma verificação desnecessária.

Como deixar passar um município em risco custa mais do que verificar um que não precisava, a métrica principal deve medir a capacidade de **encontrar os casos de Alto Risco**.

| Métrica | O que mede | Papel |
|---|---|---|
| **Recall** (Alto Risco) | Dos municípios realmente em risco, quantos o modelo achou | **Principal** |
| **Precisão** (Alto Risco) | Dos que o modelo marcou, quantos realmente eram risco | Apoio (controla os alarmes falsos) |
| **F2** | Resumo de precisão e recall, com maior peso no recall | Apoio |
| **PR-AUC** | Qualidade da separação das classes em todos os cortes | Apoio (comparação entre modelos) |

**Acurácia descartada:** com ~79% de registros "Estáveis", um classificador que sempre responde "Estável" teria ~79% de acurácia sem encontrar nenhum município em risco.

In [7]:
# Todas as métricas consideram a classe positiva = Alto Risco (y = 1)
scoring = {
    'Recall':   make_scorer(recall_score, zero_division=0),
    'Precisão': make_scorer(precision_score, zero_division=0),
    'F2':       make_scorer(fbeta_score, beta=2, zero_division=0),
    'PR-AUC':   'average_precision',
}
print('Métricas definidas:', list(scoring))

Métricas definidas: ['Recall', 'Precisão', 'F2', 'PR-AUC']


## 6.6 Modelos de referência e validação cruzada

| Modelo | O que faz | Papel |
|---|---|---|
| **Dummy** | Responde sempre "Estável" (classe majoritária) | Piso absoluto: qualquer modelo útil precisa superá-lo |
| **Regressão logística** | Modelo linear simples e interpretável | Baseline "de verdade" |

Escolhas da regressão logística:
- `class_weight='balanced'`: penaliza mais os erros na classe Alto Risco (a rara), o que favorece o recall, coerente com o custo do erro.
- Corte padrão de 0,5: por ser o baseline, nenhum ajuste fino é feito.

O pré-processamento (imputação pela mediana, padronização e codificação *one-hot* de `sigla_uf`) fica dentro de um `Pipeline`, ajustado **somente no treino**.

A comparação é feita por **validação cruzada de 5 folds dentro do treino**, com o mesmo agrupamento por município. O teste **não é usado** nesta etapa.

In [8]:
def montar_pipeline(cols_num, cols_cat, modelo):
    """Pré-processamento + modelo em um único Pipeline (evita vazamento entre treino e teste)."""
    pre = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                          ('esc', StandardScaler())]), cols_num),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cols_cat),
    ])
    return Pipeline([('pre', pre), ('modelo', modelo)])


modelos = {
    'Dummy (sempre Estável)': DummyClassifier(strategy='most_frequent'),
    'Regressão logística':    LogisticRegression(class_weight='balanced',
                                                 max_iter=1000, random_state=SEMENTE),
}

In [9]:
# Validação cruzada só no treino, agrupada por município
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
grupos_train = df_train['id_municipio']

linhas_cv = []
for nome_cen, (cols_num, cols_cat) in cenarios.items():
    for nome_mod, modelo in modelos.items():
        pipe = montar_pipeline(cols_num, cols_cat, modelo)
        res = cross_validate(pipe, df_train[cols_num + cols_cat], y_train,
                             groups=grupos_train, cv=cv, scoring=scoring)
        linha = {'Cenário': nome_cen, 'Modelo': nome_mod}
        for m in scoring:
            linha[m] = f'{res[f"test_{m}"].mean():.3f} ± {res[f"test_{m}"].std():.3f}'
        linhas_cv.append(linha)

print('=' * 70)
print('BASELINE - VALIDAÇÃO CRUZADA NO TREINO (média ± desvio, 5 folds)')
print('=' * 70)
print(pd.DataFrame(linhas_cv).to_string(index=False))

BASELINE - VALIDAÇÃO CRUZADA NO TREINO (média ± desvio, 5 folds)
        Cenário                 Modelo        Recall      Precisão            F2        PR-AUC
A (só contexto) Dummy (sempre Estável) 0.000 ± 0.000 0.000 ± 0.000 0.000 ± 0.000 0.209 ± 0.006
A (só contexto)    Regressão logística 0.783 ± 0.028 0.496 ± 0.021 0.701 ± 0.011 0.642 ± 0.017
B (+ português) Dummy (sempre Estável) 0.000 ± 0.000 0.000 ± 0.000 0.000 ± 0.000 0.209 ± 0.006
B (+ português)    Regressão logística 0.857 ± 0.010 0.591 ± 0.017 0.786 ± 0.009 0.822 ± 0.008


## 6.7 Avaliação final no conjunto de teste

Esta é a "prova final": cada modelo é treinado com **todo o treino** e avaliado no teste, que não foi tocado até aqui. A execução deve acontecer **uma única vez**, sem ajustar o modelo depois de ver o resultado, para que o teste continue honesto.

Além das métricas, as **matrizes de confusão** mostram em números absolutos os acertos e os dois tipos de erro.

In [10]:
linhas_teste = []
matrizes = {}

for nome_cen, (cols_num, cols_cat) in cenarios.items():
    cols = cols_num + cols_cat
    for nome_mod, modelo in modelos.items():
        pipe = montar_pipeline(cols_num, cols_cat, clone(modelo))
        pipe.fit(df_train[cols], y_train)                  # treina com todo o treino
        pred = pipe.predict(df_test[cols])                 # classe prevista (0 ou 1)
        prob = pipe.predict_proba(df_test[cols])[:, 1]     # probabilidade de Alto Risco

        linhas_teste.append({
            'Cenário': nome_cen,
            'Modelo': nome_mod,
            'Recall': recall_score(y_test, pred, zero_division=0),
            'Precisão': precision_score(y_test, pred, zero_division=0),
            'F2': fbeta_score(y_test, pred, beta=2, zero_division=0),
            'PR-AUC': average_precision_score(y_test, prob),
        })
        matrizes[(nome_cen, nome_mod)] = confusion_matrix(y_test, pred)

print('=' * 70)
print('AVALIAÇÃO FINAL NO CONJUNTO DE TESTE')
print('=' * 70)
print(pd.DataFrame(linhas_teste).round(3).to_string(index=False))

print('\n' + '=' * 70)
print('MATRIZ DE CONFUSÃO - REGRESSÃO LOGÍSTICA (teste)')
print('=' * 70)
for (cen, mod), cm in matrizes.items():
    if 'logística' in mod:
        tn, fp, fn, tp = cm.ravel()
        print(f'\nCenário {cen}')
        print(f'   Alto Risco achados corretamente (verdadeiros positivos): {tp}')
        print(f'   Alto Risco que passaram batido (falsos negativos)      : {fn}')
        print(f'   Alarmes falsos (falsos positivos)                      : {fp}')
        print(f'   Estáveis classificados corretamente (verdadeiros neg.) : {tn}')

AVALIAÇÃO FINAL NO CONJUNTO DE TESTE
        Cenário                 Modelo  Recall  Precisão    F2  PR-AUC
A (só contexto) Dummy (sempre Estável)   0.000     0.000 0.000   0.207
A (só contexto)    Regressão logística   0.759     0.487 0.683   0.625
B (+ português) Dummy (sempre Estável)   0.000     0.000 0.000   0.207
B (+ português)    Regressão logística   0.868     0.606 0.799   0.831

MATRIZ DE CONFUSÃO - REGRESSÃO LOGÍSTICA (teste)

Cenário A (só contexto)
   Alto Risco achados corretamente (verdadeiros positivos): 749
   Alto Risco que passaram batido (falsos negativos)      : 238
   Alarmes falsos (falsos positivos)                      : 788
   Estáveis classificados corretamente (verdadeiros neg.) : 2988

Cenário B (+ português)
   Alto Risco achados corretamente (verdadeiros positivos): 857
   Alto Risco que passaram batido (falsos negativos)      : 130
   Alarmes falsos (falsos positivos)                      : 557
   Estáveis classificados corretamente (verdadeiros neg.) :

## 6.8 Conclusões e limitações

**Resultados no teste** (regressão logística comparada ao classificador trivial; valores obtidos com `SEMENTE = 42`)

| Cenário | Modelo | Recall | Precisão | F2 | PR-AUC |
|---|---|---|---|---|---|
| A (só contexto) | Dummy (sempre Estável) | 0,000 | 0,000 | 0,000 | 0,207 |
| A (só contexto) | **Regressão logística** | **0,759** | 0,487 | 0,683 | 0,625 |
| B (+ português) | Dummy (sempre Estável) | 0,000 | 0,000 | 0,000 | 0,207 |
| B (+ português) | **Regressão logística** | **0,868** | 0,606 | 0,799 | 0,831 |

**Leitura**
- O classificador trivial tem recall 0 e PR-AUC igual à proporção de Alto Risco (~0,21), o nível do acaso. A regressão logística supera esse piso nos dois cenários.
- Os resultados do teste ficaram muito próximos aos da validação cruzada, sem sinal de sobreajuste ou vazamento.
- **Cenário A:** o modelo encontrou 749 dos 987 registros de Alto Risco (238 falsos negativos e 788 falsos positivos).
- **Cenário B:** encontrou 857 dos 987 (130 falsos negativos e 557 falsos positivos). O ganho é grande, mas deve ser lido como **limite superior**, pois a nota de Português acompanha a de Matemática, que define o alvo.

**Limitações**
- "Risco" aqui é o risco do **município** (desempenho e rendimento abaixo dos limiares), e não a evasão de alunos individuais. Os registros são **município-ano**.
- O corte de classificação (0,5) não foi otimizado, e o recall alto vem ao custo de uma precisão de ~50% a 60%, coerente com a prioridade de não deixar casos críticos sem identificação.
- Os limiares do alvo (250 pontos e 0,75 de rendimento) foram definidos na análise exploratória e condicionam todos os resultados.

**Próximos passos:** ajustar o corte de decisão e testar modelos mais flexíveis (por exemplo, árvores e *ensembles*), sempre comparando com este baseline sob o mesmo protocolo.